# 02 · EDA and leakage-safe feature engineering

Reads the TRAIN split and the tables/figures written by `python -m ssn eda`. All statistics are computed on the training split only; the test split is never opened here.

Dataset: UCI 697 (CC BY 4.0). Governing rules: `.specify/memory/constitution.md` (Principle V leakage, X fairness).

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from ssn.features import allowlist as al
from ssn.features import engineering as E
from ssn.paths import repo_root

ROOT = repo_root()
T, F = ROOT / 'reports' / 'tables', ROOT / 'reports' / 'figures'
allow = al.load(ROOT / 'configs' / 'features.yaml')
train = pd.read_parquet(ROOT / 'data' / 'processed' / 'train.parquet')
print('train rows:', len(train), '| is_dropout rate:', round(train.is_dropout.mean(), 4))
pd.read_csv(T / 'split_summary.csv')

## 1. Cleaning: before/after counts

No rows were removed; the treatments and their counts are logged below. Outliers are flagged, not removed.

In [ ]:
pd.read_csv(T / 'clean_before_after.csv')

## 2. Target distribution (train)

In [ ]:
display(Image(str(F / 'eda_target_distribution.png')))

## 3. Feature-availability list (auditable)

The allow-list is the single source of truth for what the deployed model may see.

In [ ]:
summary = pd.DataFrame([{'column': c, 'availability': m['availability'], 'role': m['role'], 'dtype': m['dtype'],
                         'allowed_model_input': c in allow.allowed} for c, m in allow.columns.items()])
print(summary.groupby(['availability', 'role']).size())
summary[~summary.allowed_model_input]

## 4. Engineered first-semester features

Stateless, row-wise, computed from allow-listed inputs only. Zero denominators become NaN and are imputed later inside the CV pipeline.

In [ ]:
pd.DataFrame([{'feature': n, 'inputs': ', '.join(i), 'rationale': r} for n, (i, r) in E.FEATURE_SPECS.items()])

In [ ]:
display(pd.read_csv(T / 'zero_denominators.csv'))
eng = E.engineer(train)
eng.describe().T

## 5. Distributions by outcome

In [ ]:
display(Image(str(F / 'eda_numeric_source_by_target.png')))
display(Image(str(F / 'eda_engineered_by_target.png')))

In [ ]:
pd.read_csv(T / 'eda_numeric_summary_by_target.csv').pivot(index='feature', columns='is_dropout', values=['mean', 'median'])

## 6. Categorical features: dropout rate by category (groups with n ≥ min_group_size)

In [ ]:
display(Image(str(F / 'eda_dropout_rate_by_category.png')))
cat = pd.read_csv(T / 'eda_dropout_rate_by_category.csv')
cat[cat.reliable].sort_values('dropout_rate', ascending=False).head(15)

## 7. Correlations (Spearman) with the target

In [ ]:
display(Image(str(F / 'eda_correlation_spearman.png')))
pd.read_csv(T / 'eda_spearman_with_target.csv', index_col=0)

## 8. Sensitive attributes: aggregate distributions (audit context only)

These columns are **never model inputs**. They are shown here to understand group sizes and base rates before the fairness audit. Grey bars mark groups below the minimum reliable size.

In [ ]:
display(Image(str(F / 'eda_sensitive_groups.png')))
sens = pd.read_csv(T / 'eda_dropout_rate_by_sensitive_group.csv')
sens[sens.reliable]

In [ ]:
display(pd.read_csv(T / 'eda_age_summary.csv'))
display(pd.read_csv(T / 'eda_age_band_alternatives.csv'))
pd.read_csv(T / 'eda_age_band_counts.csv')

## 9. ANALYSIS ONLY: second-semester columns

Shown to document why they are prohibited: they describe the period *after* the prediction point. They are excluded from every model input path by `configs/features.yaml` and enforced by tests.

In [ ]:
display(Image(str(F / 'analysis_only_second_semester_distributions.png')))